# Week 1: 멀티모달 LLM - EMR 스캔 이미지에서 텍스트 추출

GPT-5.4-nano의 비전(Vision) 기능을 활용하여 EMR 스캔 이미지에서 텍스트를 추출합니다.

## 파이프라인
```
EMR 스캔 이미지 → GPT-5.4-nano (비전) → 추출된 텍스트
```

## 핵심 개념
- **멀티모달 LLM**: 텍스트뿐 아니라 이미지도 입력으로 받을 수 있는 LLM
- **기존 OCR과의 차이**: Tesseract 등 전통 OCR은 글자만 인식하지만, 멀티모달 LLM은 문서의 맥락과 구조까지 이해

## 사용법
1. 셀을 위에서부터 순서대로 실행 (GPU 불필요)
2. 셀 2에서 본인의 OpenAI API 키 입력

---

## 1. 패키지 설치

In [ ]:
!pip install -q openai pillow

print("설치 완료!")

## 2. API 키 설정 & 샘플 데이터 준비

In [ ]:
import os
from getpass import getpass

# OpenAI API 키 입력
api_key = getpass("OpenAI API Key를 입력하세요: ")
os.environ["OPENAI_API_KEY"] = api_key
print("API 키 설정 완료!")

# GitHub에서 샘플 EMR 이미지 다운로드
!git clone --depth 1 https://github.com/{계정}/{레포}.git /content/repo 2>/dev/null || true
!cp /content/repo/emr_samples/*.png /content/ 2>/dev/null || true

# 또는 Google Drive에서 다운로드
# !pip install -q gdown
# !gdown --folder "폴더ID" -O /content/emr_samples/ --remaining-ok

import glob
samples = sorted(glob.glob("/content/*.png"))
print(f"\n샘플 이미지 {len(samples)}개 준비 완료")
for f in samples:
    print(f"  - {os.path.basename(f)}")

## 3. EMR 스캔 이미지 확인

먼저 어떤 이미지인지 눈으로 확인해봅시다.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# 첫 번째 샘플 이미지 표시
img_path = samples[0]
img = Image.open(img_path)

plt.figure(figsize=(10, 14))
plt.imshow(img)
plt.title(os.path.basename(img_path))
plt.axis("off")
plt.show()

print(f"이미지 크기: {img.size}")

## 4. GPT-5.4-nano에 이미지 보내기 (핵심!)

이미지를 base64로 인코딩하여 GPT-5.4-nano에 전달합니다.

### 어떻게 동작하나?
1. 이미지를 base64 문자열로 변환
2. OpenAI API에 텍스트 + 이미지를 함께 전송
3. GPT-5.4-nano가 이미지를 "보고" 텍스트를 추출

In [ ]:
import base64
from openai import OpenAI

client = OpenAI()


def encode_image(image_path):
    """이미지를 base64 문자열로 변환한다."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def extract_text_from_image(image_path, model="gpt-5.4-nano"):
    """GPT-5.4-nano에 이미지를 보내서 텍스트를 추출한다."""
    base64_image = encode_image(image_path)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "이 의료 문서 이미지에서 보이는 모든 텍스트를 정확하게 추출해주세요. "
                               "문서의 레이아웃(표, 섹션 등)을 최대한 유지하면서 추출해주세요. "
                               "OCR 오류로 보이는 부분이 있으면 문맥에 맞게 보정해주세요."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        max_tokens=4096,
    )

    return response.choices[0].message.content


print("함수 정의 완료!")

## 5. 텍스트 추출 실행

첫 번째 샘플 이미지로 텍스트를 추출해봅니다.

In [ ]:
# 첫 번째 샘플로 텍스트 추출
img_path = samples[0]
print(f"처리 중: {os.path.basename(img_path)}\n")

extracted_text = extract_text_from_image(img_path)

print("=" * 60)
print("추출된 텍스트")
print("=" * 60)
print(extracted_text)

## 6. 여러 이미지 일괄 추출

모든 샘플 이미지에 대해 텍스트를 추출합니다.

In [ ]:
results = {}

for img_path in samples:
    name = os.path.basename(img_path)
    print(f"처리 중: {name}...")
    try:
        text = extract_text_from_image(img_path)
        results[name] = text
        print(f"  → 완료 ({len(text)}자)")
    except Exception as e:
        print(f"  → 실패: {e}")
        results[name] = f"오류: {e}"

print(f"\n=== 전체 {len(results)}개 파일 처리 완료 ===")

## 7. 추출 결과 비교

각 병원별 추출 결과를 확인합니다.

In [ ]:
for name, text in results.items():
    print(f"\n{'=' * 60}")
    print(f"파일: {name}")
    print(f"{'=' * 60}")
    print(text)
    print()

## 8. (실습) 프롬프트 바꿔보기

아래 셀에서 프롬프트를 수정하여 다른 방식으로 추출해보세요.

**시도해볼 것:**
- "환자 이름과 진단명만 추출해줘"
- "이 문서를 마크다운 표로 정리해줘"
- "이 차트에서 처방된 약물 목록만 뽑아줘"

In [ ]:
# === 프롬프트를 자유롭게 수정해보세요 ===
my_prompt = "이 의료 문서에서 처방된 약물 목록만 추출해주세요. 약물명, 용량, 용법을 표로 정리해주세요."
# ========================================

img_path = samples[0]
base64_image = encode_image(img_path)

response = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": my_prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{base64_image}",
                        "detail": "high"
                    }
                }
            ]
        }
    ],
    max_tokens=4096,
)

print(response.choices[0].message.content)

---

## 정리

### 이번 주에 배운 것
- 멀티모달 LLM(GPT-5.4-nano)에 이미지를 입력하는 방법
- base64 인코딩으로 이미지를 API에 전달하는 방법
- 프롬프트에 따라 추출 결과가 달라지는 것을 확인

### 기존 OCR(Tesseract, PaddleOCR) vs 멀티모달 LLM
| | 기존 OCR | 멀티모달 LLM |
|---|---|---|
| 동작 방식 | 글자 모양을 패턴 매칭 | 문서 전체를 이해 |
| 레이아웃 | 위치 좌표만 반환 | 구조를 파악하여 정리 |
| 오타 보정 | 불가 | 문맥으로 보정 가능 |
| 비용 | 무료/저렴 | API 비용 발생 |
| 속도 | 빠름 | 상대적으로 느림 |

### 다음 주 예고
추출된 텍스트에서 **환자명, 진단명, 처방 등을 JSON으로 구조화**하는 방법을 배웁니다 (Structured Output).